# 03 · Exploratory runs from protocol v2 (Lab 7)

Generates only the exploratory arms added in `configs/protocol_v2.yaml` (`amendments_v2`) on the 633 test
items: full greedy rerun, sampled runs at temperature 0.3 and 1.0, and the contamination probe (100 items).
The frozen v1 runs are not repeated. Evaluation happens afterwards on CPU together with the main run's
outputs (`scripts/run_extras_evaluation.py`). Outputs: `/kaggle/working/raw/test/*.jsonl` (quote item
text; keep private).

In [ ]:
COMMIT = "main"
ARMS = "greedy_rerun_full,probe,sampled_t03,sampled_t10"     # cheapest first, so a time-out loses the least
import subprocess
sh = lambda c: print(subprocess.run(c, shell=True, capture_output=True, text=True, executable="/bin/bash").stdout[-6000:])

In [ ]:
sh("pip install -q vllm==0.32.0 2>&1 | tail -3; pip uninstall -y -q torchaudio torchao 2>&1 | tail -2")
sh("python -c \"import vllm, torch; from vllm import LLM, SamplingParams; print('vllm', vllm.__version__, '| torch', torch.__version__, '| LLM import OK')\" 2>&1")

In [ ]:
sh(f"rm -rf /kaggle/working/repo && git clone -q https://github.com/vibhum-phd/lab7-oncology-llm-voting.git /kaggle/working/repo "
   f"&& cd /kaggle/working/repo && git checkout -q {COMMIT} && git log -1 --format='commit %H %s' && grep -E '^(status|version)' configs/protocol_v2.yaml")
sh("cd /kaggle/working/repo && python scripts/fetch_medqa.py data/raw/medqa_4opt_hf 2>&1 | grep -v Warning")
sh("cd /kaggle/working/repo && for t in tests/test_*.py; do echo -n \"$t: \"; python -X utf8 $t 2>&1 | tail -1; done")

In [ ]:
sh(f"cd /kaggle/working/repo && python -X utf8 scripts/run_generation.py --split test --config configs/protocol_v2.yaml "
   f"--arms {ARMS} --out /kaggle/working/raw/test 2>&1 | grep -E '^\\{{|refusing|Error|error' | tail -20")
sh("cat /kaggle/working/raw/test/run_summary_v2.json")
sh("pip freeze > /kaggle/working/pip_freeze.txt; nvidia-smi --query-gpu=name,memory.used --format=csv")